<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "212"
iso_country = "Dominica"
import_portfolio = "h5_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2023
gdp = 659.310*1e6

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Dominica': 

Number of trade records present in dataset: 12232
Number of partner countries present in dataset: 103
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$262.1m
Total weight within the dataset: 301.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,212,Dominica,028,Antigua and Barbuda,Import,Sea,848420,848420 | Seals; mechanical,8484,8484 | Gaskets and similar,84,84 | Machinery and mechan,43949,"Machinery parts, non-electrical, n.e.c.",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,108.388418,NaN,108.388418,kg,2.6484,2.6484,2.503772,214.0,132.973394,0.332935,['Container'],['Container'],0.001656,0.001653,0.001654,0.001654,0.001655,0.001655,0.001733,0.003080,0.001906,0.001776,0.001781,0.001824,0.001697,0.003013,0.002010,0.001843,0.001800,0.001883,0.001665,0.003288,0.002495,0.002183,0.002018,0.001963,0.001634,0.003492,0.002873,0.002685,0.002160,0.002104,0.001603,0.003547,0.003396,0.00343,0.002336,0.00221,0.001572,0.003473,0.003316,0.003628,0.002706,0.002251,0.001542,0.003090,0.003227,0.003379,0.002515,0.002278,0.001512,0.002839,0.002883,0.003554,0.002945,0.002293,43949
1,1,212,Dominica,028,Antigua and Barbuda,Import,Sea,848490,848490 | Gasket sets or assor,8484,8484 | Gaskets and similar,84,84 | Machinery and mechan,43924,Gaskets of metal sheeting,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,51.496814,NaN,51.496814,kg,0.6621,0.6621,1.189576,214.0,132.973394,0.158182,['Container'],['Container'],0.000787,0.000786,0.000786,0.000786,0.000786,0.000786,0.000824,0.001463,0.000905,0.000844,0.000846,0.000867,0.000806,0.001432,0.000955,0.000876,0.000855,0.000894,0.000791,0.001562,0.001185,0.001037,0.000959,0.000932,0.000776,0.001659,0.001365,0.001276,0.001026,0.001000,0.000762,0.001685,0.001614,0.00163,0.001110,0.00105,0.000747,0.001650,0.001576,0.001724,0.001286,0.001070,0.000733,0.001468,0.001533,0.001605,0.001195,0.001082,0.000718,0.001349,0.001370,0.001688,0.001399,0.001090,43924


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $68.1m (26.0%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
22 | Beverages, spirits a  12640830.0  42251664.2      4.8     14.0
02 | Meat and edible meat   8016171.0   3948789.0      3.1      1.3
21 | Miscellaneous edible   6860392.6   1980319.8      2.6      0.7
19 | Preparations of cere   6667208.6   2666932.3      2.5      0.9
04 | Dairy produce; birds   5945642.3   2151093.4      2.3      0.7
11 | Products of the mill   4219055.0  10186947.2      1.6      3.4
15 | Animal, vegetable or   3643681.8   2959281.4      1.4      1.0
16 | Meat, fish, crustace   3507694.7    784001.3      1.3      0.3
20 | Preparations of vege   3485523.6   2333768.1      1.3      0.8
23 | Food industries, res   2858039.5   2114895.3      1.1      0.7
17 | Sugars and sugar con   2654911.4   4625186.3      1.0      1.5
07 | Vegetables 

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $63.4m (24.2%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Food & Beverages  35324142.4  50228133.8     13.5     16.7
Agriculture       26728799.8  26221044.9     10.2      8.7
Fishing            1329043.1    170599.7      0.5      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $53.5m (20.4%), ... 

                                                cifvalue  vol_kg_vtw  pct_val  \
gtap_desc                                                                       
OFD - Food products n.e.c.                    27674000.6  20239919.6     10.6   
OMT - Meat products n.e.c.                    10186204.5   4419537.2      3.9   
MIL - Dairy products                           6517309.1   2274321.9      2.5   
VOL - Vegetable oils and fats                  3639645.0   2957650.2      1.4   
V_F - Vegetables, fruit, nuts                  3067251.7   4656277.4      1.2   
PDR - Paddy rice                                680261.8   1181614.8      0.3   
OCR - Crops n.e.c.                              578880.2    160949.6      0.2   
CMT - Bovine meat prods                         404508.6    119299.6      0.2   
PCR - Processed rice                            387957.2    673881.6      0.1   
SGR - Sugar                                 

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 26.0% 
GTAP 20.4% 
EORA 24.2%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_23072/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,212,Dominica,028,Antigua and Barbuda,Import,Sea,848420,848420 | Seals; mechanical,8484,8484 | Gaskets and similar,84,84 | Machinery and mechan,43949,"Machinery parts, non-electrical, n.e.c.",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,108.388418,NaN,108.388418,kg,2.6484,2.6484,2.503772,214.0,132.973394,0.332935,['Container'],['Container'],0.001656,0.001653,0.001654,0.001654,0.001655,0.001655,0.001733,0.003080,0.001906,0.001776,0.001781,0.001824,0.001697,0.003013,0.002010,0.001843,0.001800,0.001883,0.001665,0.003288,0.002495,0.002183,0.002018,0.001963,0.001634,0.003492,0.002873,0.002685,0.002160,0.002104,0.001603,0.003547,0.003396,0.00343,0.002336,0.00221,0.001572,0.003473,0.003316,0.003628,0.002706,0.002251,0.001542,0.003090,0.003227,0.003379,0.002515,0.002278,0.001512,0.002839,0.002883,0.003554,0.002945,0.002293,43949,NaN
1,1,212,Dominica,028,Antigua and Barbuda,Import,Sea,848490,848490 | Gasket sets or assor,8484,8484 | Gaskets and similar,84,84 | Machinery and mechan,43924,Gaskets of metal sheeting,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,51.496814,NaN,51.496814,kg,0.6621,0.6621,1.189576,214.0,132.973394,0.158182,['Container'],['Container'],0.000787,0.000786,0.000786,0.000786,0.000786,0.000786,0.000824,0.001463,0.000905,0.000844,0.000846,0.000867,0.000806,0.001432,0.000955,0.000876,0.000855,0.000894,0.000791,0.001562,0.001185,0.001037,0.000959,0.000932,0.000776,0.001659,0.001365,0.001276,0.001026,0.001000,0.000762,0.001685,0.001614,0.00163,0.001110,0.00105,0.000747,0.001650,0.001576,0.001724,0.001286,0.001070,0.000733,0.001468,0.001533,0.001605,0.001195,0.001082,0.000718,0.001349,0.001370,0.001688,0.001399,0.001090,43924,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $40.0m of $262.1m (15.3%)
Volume: 34.0 of 301.1 million kilograms (11.3%)
Value-to-Weight: 1.18 from 0.87 US$ per kg (+35.3%)
Average distance to market (volume-weighted): 1733.8 from 1697.3 nm (+2.1%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    357  17275492.8  15767238.9      6.6      5.2
1        Meat and Prep     71   7022556.1   3708817.8      2.7      1.2
2                Dairy     78   5902242.8   2130295.3      2.3      0.7
3           Vegetables    199   3581217.1   5067387.6      1.4      1.7
4      Sugar and Honey     53   2659322.2   4626450.9      1.0      1.5
5        Fats and Oils     80   2651822.8   1792108.8      1.0      0.6
6               Fruits     84    899308.3    704865.3      0.3      0.2
7  Unprocessed Cereals     19     51533.2    192278.1      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,3595547.1,6326619.8,3939166.3,3676115.4,3686728.7,3778519.8,3330863.3,7291046.4,6969176.3,7058309.2,4813528.8,4560496.1,3146292.9,5847814.1,5940405.2,7285824.9,6038631.0,4732014.6
1,Fats and Oils,3180058.2,5327847.6,3429834.3,3211669.8,3220391.7,3319824.7,2970232.7,6097646.8,5745047.7,5844773.2,4156260.0,3932830.8,2823811.0,4950080.2,5127359.4,6169047.2,5076198.8,4064018.0
2,Fruits,911387.4,1618296.3,1001748.3,934071.7,936620.2,959106.9,842910.3,1863628.4,1783568.8,1800999.4,1226804.0,1162020.7,795111.8,1493286.3,1516279.0,1867231.4,1545198.8,1205550.0
3,Meat and Prep,5050674.4,8975692.1,5553413.7,5174895.6,5189263.7,5315619.1,4670724.1,10334801.4,9897069.4,9996174.9,6805912.5,6441214.9,4405509.1,8272135.4,8400769.7,10354725.9,8581445.6,6682252.3
4,Processed Cereals,6916487.0,11998020.4,7538638.4,7050455.1,7071810.2,7253374.5,6423340.7,13825945.3,13200813.9,13422137.1,9198009.5,8703037.0,6080267.3,11144789.0,11328509.5,13801057.9,11416821.8,9030028.9
5,Sugar and Honey,4304608.7,7604172.0,4722846.0,4404785.3,4417273.2,4526395.8,3985000.5,8756234.9,8378260.8,8475565.1,5780980.6,5471529.8,3762076.6,7020658.9,7132692.3,8766894.5,7261737.8,5676392.7
6,Unprocessed Cereals,209509.4,359928.6,227562.2,213089.4,213729.4,219419.2,194905.8,416614.3,398297.4,405261.0,279278.9,262186.3,184738.7,336497.1,341721.6,413959.4,343601.2,271641.9
7,Vegetables,7672831.3,13417331.3,8386940.1,7834727.4,7857976.1,8055813.7,7115854.6,15447430.4,14756729.9,14976180.7,10234181.5,9697909.1,6727859.0,12423088.6,12627452.9,15443900.0,12774118.9,10064263.1


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.46,0.06,0.01,0.02,0.03,0.67,0.62,0.63,0.25,0.21,0.46,0.47,0.70,0.49,0.27
1,Fats and Oils,0.81,0.09,0.01,0.02,0.05,1.18,1.05,1.08,0.45,0.36,0.80,0.87,1.26,0.85,0.47
2,Fruits,0.79,0.10,0.03,0.03,0.05,1.14,1.05,1.07,0.43,0.35,0.78,0.80,1.19,0.83,0.46
3,Meat and Prep,0.56,0.07,0.02,0.02,0.04,0.81,0.74,0.76,0.30,0.25,0.55,0.57,0.85,0.59,0.32
4,Processed Cereals,0.29,0.04,0.01,0.01,0.02,0.43,0.39,0.41,0.16,0.13,0.29,0.30,0.45,0.31,0.17
5,Sugar and Honey,1.24,0.16,0.04,0.04,0.08,1.79,1.65,1.69,0.68,0.56,1.23,1.27,1.88,1.32,0.72
6,Unprocessed Cereals,2.92,0.35,0.07,0.08,0.19,4.30,3.95,4.08,1.64,1.31,2.94,3.05,4.45,3.08,1.69
7,Vegetables,1.60,0.20,0.05,0.05,0.11,2.33,2.13,2.19,0.87,0.72,1.59,1.65,2.43,1.69,0.93
